# KOSPI 섹터별 일별 등락률 히트맵
**행 = 섹터 / 열 = 날짜(최근 1개월) / 셀 = 당일 등락률**

▶ 셀을 순서대로 실행하거나, 상단 메뉴 **런타임 → 모두 실행** 클릭

In [ ]:
# ① 패키지 설치 (최초 1회, 약 30초)
!pip install pykrx -q
!apt-get install -y fonts-nanum -q
import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)  # 폰트 캐시 갱신
print('설치 완료')

In [ ]:
# ② 날짜 설정 — 원하는 기준일로 변경하세요 (YYYYMMDD)
END_DATE = ''   # 비워두면 가장 최근 평일 자동 선택

from datetime import datetime, timedelta
if not END_DATE:
    today = datetime.today()
    if today.weekday() >= 5:
        today -= timedelta(days=today.weekday() - 4)
    END_DATE = today.strftime('%Y%m%d')
print('기준일:', END_DATE)

In [ ]:
# ③ 데이터 수집
import warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')

SECTORS = {
    '1005':'음식료품','1006':'섬유의복','1007':'종이목재','1008':'화학',
    '1009':'의약품','1010':'비금속광물','1011':'철강금속','1012':'기계',
    '1013':'전기전자','1014':'의료정밀','1015':'운수장비','1016':'유통업',
    '1017':'전기가스업','1018':'건설업','1019':'운수창고업','1020':'통신업',
    '1021':'금융업','1022':'은행','1023':'증권','1024':'보험','1025':'서비스업',
}

end_dt  = datetime.strptime(END_DATE, '%Y%m%d')
start_str = (end_dt - timedelta(days=40)).strftime('%Y%m%d')

def fetch_real():
    from pykrx import stock
    matrix = {}
    for sid, name in SECTORS.items():
        df = stock.get_index_ohlcv_by_date(start_str, END_DATE, sid)
        if df is not None and not df.empty:
            ret = df['종가'].pct_change() * 100
            matrix[name] = ret
    if not matrix:
        return None
    result = pd.DataFrame(matrix).T
    result.columns = [d.strftime('%Y-%m-%d') for d in result.columns]
    return result.iloc[:, 1:].round(2)

def make_demo():
    s, e = datetime.strptime(start_str, '%Y%m%d'), end_dt
    days = [s+timedelta(days=i) for i in range((e-s).days+1)
            if (s+timedelta(days=i)).weekday() < 5]
    cols = [d.strftime('%Y-%m-%d') for d in days]
    np.random.seed(42)
    rows = {n: np.clip((np.random.normal(.05,.7,len(cols))+
                        np.random.normal(0,1.1,len(cols))).round(2),-8,8)
            for n in SECTORS.values()}
    return pd.DataFrame(rows, index=cols).T

try:
    df = fetch_real()
    if df is not None and not df.empty:
        df = df.iloc[:, -25:]
        is_real = True
    else:
        raise ValueError
except Exception:
    df = make_demo()
    is_real = False

src = '실시간 KRX' if is_real else '데모 데이터'
print(f'데이터 소스: {src} | 섹터 {df.shape[0]}개 × {df.shape[1]}일')
df

In [ ]:
# ④ 히트맵 차트 출력
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Nanum 폰트 등록
for f in fm.findSystemFonts():
    if 'NanumGothic' in f and 'Bold' in f:
        fm.fontManager.addfont(f)
        prop = fm.FontProperties(fname=f)
        matplotlib.rcParams['font.family'] = prop.get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

def cell_color(v):
    if np.isnan(v): return (.15,.15,.15,1.)
    t = np.clip(v/4., -1, 1)
    if t > 0:   return (np.clip(1-t*.75,0,1), np.clip(1-t*.15,0,1), np.clip(1-t*.75,0,1), 1.)
    elif t < 0: return (np.clip(1+t*.15,0,1), np.clip(1+t*.85,0,1), np.clip(1+t*.85,0,1), 1.)
    return (.97,.97,.97,1.)

def text_color(bg):
    r,g,b,_ = bg
    return 'white' if .299*r+.587*g+.114*b < .55 else '#222'

n_s, n_d = df.shape
cell_w, cell_h = 2.1, 0.62
fig, ax = plt.subplots(figsize=(n_d*cell_w+3.5, n_s*cell_h+2.8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.set_xlim(0, n_d); ax.set_ylim(0, n_s); ax.axis('off')

for ri, sector in enumerate(df.index):
    y = n_s - 1 - ri
    for ci, date in enumerate(df.columns):
        val = df.at[sector, date]
        bg = cell_color(val); tc = text_color(bg)
        ax.add_patch(plt.Rectangle((ci+.02, y+.04), .96, .92,
                     facecolor=bg, edgecolor='#0d1117', linewidth=.5))
        if not np.isnan(val):
            sign = '▲' if val>0 else ('▼' if val<0 else ' ')
            ax.text(ci+.5, y+.5, f'{sign}{abs(val):.2f}%',
                    ha='center', va='center', fontsize=8.2, color=tc, fontweight='bold')
    ax.text(-.15, y+.5, sector, ha='right', va='center', fontsize=10, color='#e6edf3')

for ci, date in enumerate(df.columns):
    ax.text(ci+.5, n_s+.15, date[5:], ha='center', va='bottom',
            fontsize=8.5, color='#8b949e', rotation=45)

for c in range(0, n_d+1, 5):
    ax.axvline(c, color='#30363d', linewidth=.8)

note = '' if is_real else '  [데모 데이터]'
end_disp = f"{END_DATE[:4]}.{END_DATE[4:6]}.{END_DATE[6:]}"
fig.text(.5, .98, f'KOSPI 섹터별 일별 등락률  |  최근 1개월  (기준일 {end_disp}){note}',
         ha='center', va='top', fontsize=15, fontweight='bold', color='#e6edf3')

for pct, label in [(-4,'-4%↓'),(-2,'-2%'),(0,'0%'),(2,'+2%'),(4,'+4%↑')]:
    bg = cell_color(float(pct))
    fig.text(.5+pct*.018, .012, label, ha='center', va='bottom', fontsize=8.5,
             color=text_color(bg),
             bbox=dict(facecolor=bg, edgecolor='none', boxstyle='round,pad=0.3'))

plt.subplots_adjust(left=.10, right=.98, top=.93, bottom=.05)
plt.show()